# IPPO Baseline — Guarded Territory

Independent PPO (no communication) on the same Guarded Territory environment used by GNN-MAPPO.
Each defender gets its own actor/critic — no GNN, no message passing.
This serves as the baseline to show how much GNN communication helps.

**Changes from navigation version:**
- Replaced `VMASAdapter` + VMAS navigation with `GuardedTerritoryAdapter` + Guarded Territory
- `ActorNetwork`: `Categorical` → `Normal` (continuous 2D actions)
- `RolloutBuffer.get()`: actions dtype `torch.long` → `torch.float32`
- `PPOAgent.select_action`: returns `(2,)` action tensor, not scalar
- `IPPOTrainer`: tensor-based loop via adapter, no dict conversion
- Added evaluation function for the new environment

In [ ]:
%pip -q install vmas matplotlib

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

import typing
import vmas
from vmas.simulator.core import Agent, World, Landmark, Sphere
from vmas.simulator.scenario import BaseScenario
from vmas.simulator.utils import Color

def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {get_device()}")

### Guarded Territory Scenario + Adapter

Identical to the GNN-MAPPO notebook — included here so this notebook is self-contained.

In [ ]:
# ── Agent Types ─────────────────────────────────────────────
SCOUT = "scout"
INTERCEPTOR = "interceptor"
INTRUDER = "intruder"


class Scenario(BaseScenario):
    def make_world(self, batch_dim: int, device: torch.device, **kwargs):
        self.n_scouts = kwargs.get("n_scouts", 3)
        self.n_interceptors = kwargs.get("n_interceptors", 3)
        self.n_intruders = kwargs.get("n_intruders", 3)
        self.n_zones = kwargs.get("n_zones", 2)
        self.world_size = kwargs.get("world_size", 5.0)
        self.scout_fov = kwargs.get("scout_fov", 1.0)
        self.interceptor_fov = kwargs.get("interceptor_fov", 0.5)
        self.tag_radius = kwargs.get("tag_radius", 0.1)
        self.intruder_speed = kwargs.get("intruder_speed", 0.5)
        self.defender_speed = kwargs.get("defender_speed", 0.8)
        self.n_defenders = self.n_scouts + self.n_interceptors

        world = World(batch_dim=batch_dim, device=device, dt=0.1, drag=0.25, dim_c=0,
                      x_semidim=self.world_size, y_semidim=self.world_size)

        self.scouts = []
        for i in range(self.n_scouts):
            a = Agent(name=f"scout_{i}", collide=True, mass=1.0, shape=Sphere(radius=0.075),
                      max_speed=self.defender_speed, color=Color.BLUE, u_range=1.0)
            a.agent_type, a.type_id = SCOUT, 0
            world.add_agent(a); self.scouts.append(a)

        self.interceptors = []
        for i in range(self.n_interceptors):
            a = Agent(name=f"interceptor_{i}", collide=True, mass=1.0, shape=Sphere(radius=0.09),
                      max_speed=self.defender_speed, color=Color.GREEN, u_range=1.0)
            a.agent_type, a.type_id = INTERCEPTOR, 1
            world.add_agent(a); self.interceptors.append(a)

        self.intruders = []
        for i in range(self.n_intruders):
            a = Agent(name=f"intruder_{i}", collide=True, mass=1.0, shape=Sphere(radius=0.075),
                      max_speed=self.intruder_speed, color=Color.RED, u_range=1.0)
            a.agent_type, a.type_id = INTRUDER, 2
            world.add_agent(a); self.intruders.append(a)

        self.defenders = self.scouts + self.interceptors
        self.zones = []
        for i in range(self.n_zones):
            z = Landmark(name=f"zone_{i}", collide=False, movable=False,
                         shape=Sphere(radius=0.2), color=Color.LIGHT_GREEN)
            world.add_landmark(z); self.zones.append(z)

        self._intruder_tagged = self._zone_breached = self._tag_count = None
        return world

    def reset_world_at(self, env_index=None):
        b, d = self.world.batch_dim, self.world.device
        if env_index is None:
            self._intruder_tagged = torch.zeros(b, self.n_intruders, dtype=torch.bool, device=d)
            self._zone_breached = torch.zeros(b, self.n_zones, dtype=torch.bool, device=d)
            self._tag_count = torch.zeros(b, dtype=torch.float32, device=d)
        else:
            self._intruder_tagged[env_index] = False
            self._zone_breached[env_index] = False
            self._tag_count[env_index] = 0.0

        for i, zone in enumerate(self.zones):
            pos = torch.zeros((1,2) if env_index is not None else (b,2), dtype=torch.float32, device=d)
            ang = 2*torch.pi*i/self.n_zones
            pos[...,0] = 0.3*torch.cos(torch.tensor(ang))
            pos[...,1] = 0.3*torch.sin(torch.tensor(ang))
            zone.set_pos(pos + 0.1*torch.randn_like(pos), batch_index=env_index)

        for i, df in enumerate(self.defenders):
            pos = torch.zeros((1,2) if env_index is not None else (b,2), dtype=torch.float32, device=d)
            ang = 2*torch.pi*i/self.n_defenders
            r = 0.5+0.2*torch.rand(pos.shape[0],1,device=d)
            pos[...,0:1] = r*torch.cos(torch.tensor(ang))
            pos[...,1:2] = r*torch.sin(torch.tensor(ang))
            df.set_pos(pos + 0.05*torch.randn_like(pos), batch_index=env_index)

        for i, intr in enumerate(self.intruders):
            pos = torch.zeros((1,2) if env_index is not None else (b,2), dtype=torch.float32, device=d)
            ang = 2*torch.pi*i/self.n_intruders + torch.pi
            pos[...,0] = self.world_size*0.85*torch.cos(torch.tensor(ang))
            pos[...,1] = self.world_size*0.85*torch.sin(torch.tensor(ang))
            intr.set_pos(pos + 0.1*torch.randn_like(pos), batch_index=env_index)

    def _get_intruder_actions(self, intruder):
        d, b = self.world.device, self.world.batch_dim
        min_dist = torch.full((b,), float("inf"), device=d)
        tgt = self.zones[0].state.pos.clone()
        for z in self.zones:
            dist = torch.linalg.vector_norm(intruder.state.pos - z.state.pos, dim=-1)
            closer = dist < min_dist
            min_dist = torch.where(closer, dist, min_dist)
            tgt = torch.where(closer.unsqueeze(-1), z.state.pos, tgt)
        direction = tgt - intruder.state.pos
        direction = direction / (torch.linalg.vector_norm(direction, dim=-1, keepdim=True) + 1e-6)
        return self.intruder_speed * (direction + 0.2*torch.randn(b,2,device=d))

    def process_action(self, agent):
        if hasattr(agent, "agent_type") and agent.agent_type == INTRUDER:
            agent.action.u = self._get_intruder_actions(agent)

    def observation(self, agent):
        b, d = self.world.batch_dim, self.world.device
        if hasattr(agent, "agent_type") and agent.agent_type == SCOUT:
            fov = self.scout_fov; type_oh = torch.tensor([1.,0.], device=d).expand(b,2)
        elif hasattr(agent, "agent_type") and agent.agent_type == INTERCEPTOR:
            fov = self.interceptor_fov; type_oh = torch.tensor([0.,1.], device=d).expand(b,2)
        else:
            return torch.zeros(b, 2, device=d)
        parts = [agent.state.vel, agent.state.pos, type_oh]
        for z in self.zones: parts.append(z.state.pos - agent.state.pos)
        for intr in self.intruders:
            rp = intr.state.pos - agent.state.pos
            vis = (torch.linalg.vector_norm(rp, dim=-1, keepdim=True) <= fov).float()
            parts.extend([rp*vis, intr.state.vel*vis])
        for o in self.defenders:
            if o is agent: continue
            rp = o.state.pos - agent.state.pos
            vis = (torch.linalg.vector_norm(rp, dim=-1, keepdim=True) <= fov).float()
            parts.append(rp*vis)
        return torch.cat(parts, dim=-1)

    def reward(self, agent):
        if hasattr(agent, "agent_type") and agent.agent_type == INTRUDER:
            return torch.zeros(self.world.batch_dim, device=self.world.device)
        b, d = self.world.batch_dim, self.world.device
        rew = torch.zeros(b, device=d)
        for j, intr in enumerate(self.intruders):
            if self._intruder_tagged is None: break
            at = self._intruder_tagged[:, j]
            for ic in self.interceptors:
                dist = torch.linalg.vector_norm(ic.state.pos - intr.state.pos, dim=-1)
                jt = (~at) & (dist < self.tag_radius)
                self._intruder_tagged[:, j] = self._intruder_tagged[:, j] | jt
                self._tag_count += jt.float()
        for k, zone in enumerate(self.zones):
            for j, intr in enumerate(self.intruders):
                if self._intruder_tagged is None: break
                tg = self._intruder_tagged[:, j]
                dz = torch.linalg.vector_norm(intr.state.pos - zone.state.pos, dim=-1)
                br = (~tg) & (dz < 0.15)
                nb = br & (~self._zone_breached[:, k])
                self._zone_breached[:, k] = self._zone_breached[:, k] | br
                rew -= 5.0 * nb.float()
        for j, intr in enumerate(self.intruders):
            if self._intruder_tagged is None: break
            tg = self._intruder_tagged[:, j]
            for ic in self.interceptors:
                dist = torch.linalg.vector_norm(ic.state.pos - intr.state.pos, dim=-1)
                rew += 3.0 * ((~tg) & (dist < self.tag_radius)).float()
        if hasattr(agent, "agent_type") and agent.agent_type == SCOUT:
            for intr in self.intruders:
                rew += 0.1*(torch.linalg.vector_norm(agent.state.pos-intr.state.pos,dim=-1)<self.scout_fov).float()
            for ic in self.interceptors:
                dd = torch.linalg.vector_norm(agent.state.pos-ic.state.pos,dim=-1)
                rew += 0.05*((dd>0.2)&(dd<1.0)).float()
        elif hasattr(agent, "agent_type") and agent.agent_type == INTERCEPTOR:
            md = torch.full((b,), float("inf"), device=d)
            for j, intr in enumerate(self.intruders):
                tg = self._intruder_tagged[:,j] if self._intruder_tagged is not None else torch.zeros(b,dtype=torch.bool,device=d)
                dd = torch.linalg.vector_norm(agent.state.pos-intr.state.pos,dim=-1)
                md = torch.minimum(md, torch.where(tg, torch.tensor(float("inf"),device=d), dd))
            rew -= 0.1*torch.clamp(md, max=5.0)
        return rew

    def done(self):
        if self._intruder_tagged is None or self._zone_breached is None:
            return torch.zeros(self.world.batch_dim, dtype=torch.bool, device=self.world.device)
        return self._intruder_tagged.all(dim=-1) | self._zone_breached.all(dim=-1)

    def info(self, agent): return {}


def get_obs_dim(n_scouts=3, n_interceptors=3, n_intruders=3, n_zones=2):
    nd = n_scouts + n_interceptors
    return 2+2+2 + n_zones*2 + n_intruders*2 + n_intruders*2 + (nd-1)*2


class GuardedTerritoryAdapter:
    def __init__(self, num_envs=1, device="cpu", n_scouts=3, n_interceptors=3,
                 n_intruders=3, n_zones=2, max_steps=200, **kwargs):
        self.num_envs = num_envs
        self.device = device
        self.n_scouts = n_scouts
        self.n_interceptors = n_interceptors
        self.n_defenders = n_scouts + n_interceptors
        self.env = vmas.make_env(scenario=Scenario(), num_envs=num_envs, device=device,
                                 continuous_actions=True, max_steps=max_steps,
                                 n_scouts=n_scouts, n_interceptors=n_interceptors,
                                 n_intruders=n_intruders, n_zones=n_zones, **kwargs)
        self.obs_dim = get_obs_dim(n_scouts, n_interceptors, n_intruders, n_zones)
        self.defender_indices = [i for i, a in enumerate(self.env.agents)
                                 if hasattr(a,"agent_type") and a.agent_type in (SCOUT,INTERCEPTOR)]

    def reset(self):
        all_obs = self.env.reset()
        obs = torch.stack([all_obs[i] for i in self.defender_indices], dim=1)
        return obs, obs[:,:,2:4].clone()

    def step(self, defender_actions):
        acts = []
        for i in range(len(self.env.agents)):
            if i in self.defender_indices:
                acts.append(defender_actions[:, self.defender_indices.index(i)])
            else:
                acts.append(torch.zeros(self.num_envs, 2, device=self.device))
        all_obs, all_rew, dones, all_info = self.env.step(acts)
        obs = torch.stack([all_obs[i] for i in self.defender_indices], dim=1)
        rew = torch.stack([all_rew[i] for i in self.defender_indices], dim=1)
        return obs, rew, dones, {}, obs[:,:,2:4].clone()

    @property
    def action_dim(self): return 2
    @property
    def n_agents(self): return self.n_defenders

### Actor Network (Continuous Actions)

**Changed from original:** `Categorical` → `Normal` distribution with learnable `log_std`.

In [ ]:
class ActorNetwork(nn.Module):
    def __init__(self, obs_dim, hidden_dim, action_dim, device=None):
        super().__init__()
        self.device = torch.device(device) if device else get_device()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)  # outputs mean
        self.log_std = nn.Parameter(torch.zeros(action_dim))  # learnable
        self._init_weights()
        self.to(self.device)

    def _init_weights(self):
        nn.init.orthogonal_(self.fc1.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc2.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc3.weight, gain=0.01)
        for layer in [self.fc1, self.fc2, self.fc3]:
            nn.init.constant_(layer.bias, 0.0)

    def forward(self, obs):
        if not torch.is_tensor(obs):
            obs = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
        else:
            obs = obs.to(self.device, dtype=torch.float32)
        x = F.relu(self.fc1(obs))
        x = F.relu(self.fc2(x))
        return self.fc3(x)  # mean

    def get_action_and_log_probs(self, obs, action=None):
        mean = self.forward(obs)
        std = self.log_std.exp().expand_as(mean)
        dist = Normal(mean, std)
        if action is None:
            action = dist.sample()
        else:
            if not torch.is_tensor(action):
                action = torch.as_tensor(action, dtype=torch.float32, device=self.device)
            else:
                action = action.to(self.device, dtype=torch.float32)
        log_prob = dist.log_prob(action).sum(dim=-1)  # sum over action dims
        entropy = dist.entropy().sum(dim=-1)
        return action, log_prob, entropy

    def evaluate_actions(self, obs, actions):
        if not torch.is_tensor(actions):
            actions = torch.as_tensor(actions, dtype=torch.float32, device=self.device)
        else:
            actions = actions.to(self.device, dtype=torch.float32)
        mean = self.forward(obs)
        std = self.log_std.exp().expand_as(mean)
        dist = Normal(mean, std)
        log_prob = dist.log_prob(actions).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)
        return log_prob, entropy, mean

### Critic Network

Per-agent critic (identical to original — no changes needed).

In [ ]:
class CriticNetwork(nn.Module):
    def __init__(self, obs_dim, hidden_dim, device=None):
        super().__init__()
        self.device = torch.device(device) if device else get_device()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)
        self._init_weights()
        self.to(self.device)

    def _init_weights(self):
        nn.init.orthogonal_(self.fc1.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc2.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc3.weight, gain=1.0)
        for layer in [self.fc1, self.fc2, self.fc3]:
            nn.init.constant_(layer.bias, 0.0)

    def forward(self, obs):
        if not torch.is_tensor(obs):
            obs = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
        else:
            obs = obs.to(self.device, dtype=torch.float32)
        x = F.relu(self.fc1(obs))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

### Rollout Buffer

**Fix:** `get()` now stores actions as `torch.float32` (was `torch.long` for discrete).
Actions are stored as numpy arrays of shape `(action_dim,)` per timestep.

In [ ]:
class RolloutBuffer:
    def __init__(self, buffer_size, obs_dim, action_dim, gamma, gae_lambda, device):
        self.buffer_size = buffer_size
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.device = device
        self.obs = []
        self.actions = []    # now stores (action_dim,) arrays
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []

    def add_rollout(self, obs, action, reward, done, log_prob, value):
        self.obs.append(obs)
        self.actions.append(action)  # (action_dim,) numpy or tensor
        self.rewards.append(reward.item() if torch.is_tensor(reward) else reward)
        self.dones.append(done.item() if torch.is_tensor(done) else done)
        self.log_probs.append(log_prob.item() if torch.is_tensor(log_prob) else log_prob)
        self.values.append(value.item() if torch.is_tensor(value) else value)

    def compute_returns_and_advantages(self, last_value):
        if len(self.rewards) == 0:
            raise RuntimeError("Buffer empty")
        rewards = torch.as_tensor(self.rewards, dtype=torch.float32, device=self.device)
        values = torch.as_tensor(self.values, dtype=torch.float32, device=self.device)
        dones = torch.as_tensor(self.dones, dtype=torch.float32, device=self.device)
        T = rewards.shape[0]
        advantages = torch.zeros(T, dtype=torch.float32, device=self.device)
        last_gae = 0.0
        for t in reversed(range(T)):
            nv = (torch.tensor(last_value, dtype=torch.float32, device=self.device)
                  if not torch.is_tensor(last_value)
                  else last_value.to(self.device, dtype=torch.float32)) if t == T-1 else values[t+1]
            delta = rewards[t] + self.gamma*(1-dones[t])*nv - values[t]
            advantages[t] = delta + self.gamma*self.gae_lambda*(1-dones[t])*last_gae
            last_gae = advantages[t]
        self.advantages = advantages
        self.returns = advantages + values

    def get(self):
        obs = torch.as_tensor(np.asarray(self.obs), dtype=torch.float32, device=self.device)
        actions = torch.as_tensor(np.asarray(self.actions), dtype=torch.float32, device=self.device)  # [FIX]
        log_probs = torch.as_tensor(self.log_probs, dtype=torch.float32, device=self.device)
        adv = self.advantages
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        return obs, actions, log_probs, adv, self.returns

    def clear(self):
        self.obs = []; self.actions = []; self.rewards = []
        self.dones = []; self.log_probs = []; self.values = []

### PPO Agent

**Changed:** `select_action` returns `(action_dim,)` numpy array instead of scalar.

In [ ]:
class PPOAgent:
    def __init__(self, obs_dim, hidden_dim, action_dim,
                 lr=3e-4, buffer_size=2048, gamma=0.99, gae_lambda=0.95,
                 clip_epsilon=0.2, value_coef=0.5, entropy_coef=0.01, max_grad_norm=0.5):
        self.device = get_device()
        self.actor = ActorNetwork(obs_dim, hidden_dim, action_dim, device=self.device)
        self.critic = CriticNetwork(obs_dim, hidden_dim, device=self.device)
        self.buffer = RolloutBuffer(buffer_size, obs_dim, action_dim, gamma, gae_lambda, self.device)
        self.actor_optim = torch.optim.Adam(self.actor.parameters(), lr=lr)
        self.critic_optim = torch.optim.Adam(self.critic.parameters(), lr=lr)
        self.clip_eps = clip_epsilon
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm
        self.gamma = gamma

    def select_action(self, obs):
        """Returns (action_np, log_prob_scalar, value_scalar)"""
        if not torch.is_tensor(obs):
            obs = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
        else:
            obs = obs.to(self.device, dtype=torch.float32)
        with torch.no_grad():
            action, log_prob, _ = self.actor.get_action_and_log_probs(obs)
            value = self.critic(obs)
        # Return numpy action (action_dim,), scalar log_prob, scalar value
        return action.cpu().numpy(), log_prob.item(), value.squeeze().item()

    def update(self, last_obs, num_epochs=30):
        if len(self.buffer.rewards) == 0:
            return {"policy_loss": 0., "value_loss": 0., "entropy": 0., "mean_bellman_error": 0.}

        with torch.no_grad():
            lo = torch.as_tensor(last_obs, dtype=torch.float32, device=self.device).unsqueeze(0)
            last_value = self.critic(lo).squeeze()

        self.buffer.compute_returns_and_advantages(last_value)
        obs, actions, old_lp, advantages, returns = self.buffer.get()

        tot_pi, tot_v, tot_ent = 0., 0., 0.
        for _ in range(num_epochs):
            new_lp, entropy, _ = self.actor.evaluate_actions(obs, actions)
            ratio = torch.exp(new_lp - old_lp)
            s1 = ratio * advantages
            s2 = torch.clamp(ratio, 1-self.clip_eps, 1+self.clip_eps) * advantages
            pi_loss = -torch.min(s1, s2).mean()
            vals = self.critic(obs).squeeze(-1)
            v_loss = F.mse_loss(vals, returns)
            ent_loss = -entropy.mean()
            loss = pi_loss + self.value_coef*v_loss + self.entropy_coef*ent_loss

            self.actor_optim.zero_grad()
            self.critic_optim.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.actor.parameters(), self.max_grad_norm)
            nn.utils.clip_grad_norm_(self.critic.parameters(), self.max_grad_norm)
            self.actor_optim.step()
            self.critic_optim.step()
            tot_pi += pi_loss.item(); tot_v += v_loss.item(); tot_ent += entropy.mean().item()

        with torch.no_grad():
            rv = torch.as_tensor(self.buffer.rewards, dtype=torch.float32, device=self.device)
            vv = torch.as_tensor(self.buffer.values, dtype=torch.float32, device=self.device)
            dd = torch.as_tensor(self.buffer.dones, dtype=torch.float32, device=self.device)
            nv = torch.zeros_like(vv)
            if len(vv)>1: nv[:-1] = vv[1:]
            nv[-1] = last_value
            bellman = (rv + self.gamma*(1-dd)*nv - vv).abs().mean().item()

        self.buffer.clear()
        return {"policy_loss": tot_pi/num_epochs, "value_loss": tot_v/num_epochs,
                "entropy": tot_ent/num_epochs, "mean_bellman_error": bellman}

### IPPO Trainer (Rewritten for VMAS Adapter)

**Key changes:**
- Uses `GuardedTerritoryAdapter` instead of PettingZoo dict-based env
- Defender agents indexed by integer (0..N-1) instead of string keys
- `collect_rollouts` works with tensors: squeeze batch dim, index per-agent
- VMAS auto-resets done envs

In [ ]:
class IPPOTrainer:
    def __init__(self, adapter: GuardedTerritoryAdapter, hidden_dim=64):
        self.adapter = adapter
        self.num_agents = adapter.n_defenders
        obs_dim = adapter.obs_dim
        action_dim = adapter.action_dim

        # One independent PPO agent per defender
        self.agents = [
            PPOAgent(obs_dim, hidden_dim, action_dim)
            for _ in range(self.num_agents)
        ]

        self.metrics_history = {
            "policy_loss": [], "value_loss": [], "entropy": [],
            "mean_bellman_error": [], "mean_episode_return": [], "mean_episode_rewards": [],
        }
        self._running_episode_return = 0.0

    def _safe_mean(self, values):
        return float(sum(values)/len(values)) if values else 0.0

    def collect_rollouts(self, num_steps, current_obs):
        """
        current_obs: (N, obs_dim) tensor — per-agent observations.
        Returns: (last_obs, rollout_metrics)
        """
        step_mean_rewards = []
        completed_episode_returns = []

        for _ in range(num_steps):
            # Each agent independently selects action from its own obs
            actions_np = []   # list of (action_dim,) numpy arrays
            log_probs = []    # scalars
            values = []       # scalars

            for i in range(self.num_agents):
                obs_i = current_obs[i]  # (obs_dim,) tensor
                act, lp, val = self.agents[i].select_action(obs_i)
                actions_np.append(act)
                log_probs.append(lp)
                values.append(val)

            # Stack actions into (1, N, 2) for adapter
            actions_tensor = torch.as_tensor(
                np.stack(actions_np), dtype=torch.float32,
                device=self.adapter.device,
            ).unsqueeze(0)  # (1, N, 2)

            next_obs_b, rewards_b, dones_b, _, _ = self.adapter.step(actions_tensor)

            next_obs = next_obs_b.squeeze(0)     # (N, obs_dim)
            rewards = rewards_b.squeeze(0)        # (N,)
            done_flag = dones_b.squeeze(0).item() # scalar bool

            # Store per-agent transitions
            for i in range(self.num_agents):
                self.agents[i].buffer.add_rollout(
                    current_obs[i].cpu().numpy(),   # obs
                    actions_np[i],                   # action (action_dim,)
                    rewards[i].item(),               # reward scalar
                    float(done_flag),                # done (broadcast)
                    log_probs[i],                    # log_prob scalar
                    values[i],                       # value scalar
                )

            mean_rew = rewards.mean().item()
            step_mean_rewards.append(mean_rew)
            self._running_episode_return += mean_rew

            if done_flag:
                completed_episode_returns.append(self._running_episode_return)
                self._running_episode_return = 0.0

            current_obs = next_obs

        metrics = {
            "mean_episode_return": self._safe_mean(completed_episode_returns)
            if completed_episode_returns else float(self._running_episode_return),
            "mean_episode_rewards": self._safe_mean(step_mean_rewards),
        }
        return current_obs, metrics

    def train(self, total_timesteps, rollout_length, log_every=10):
        # Initial reset
        obs_b, _ = self.adapter.reset()
        obs = obs_b.squeeze(0)  # (N, obs_dim)

        steps_done = 0
        iteration = 0

        while steps_done < total_timesteps:
            n = min(rollout_length, total_timesteps - steps_done)
            last_obs, rollout_m = self.collect_rollouts(n, obs)

            # Update each agent independently
            all_m = []
            for i in range(self.num_agents):
                m = self.agents[i].update(last_obs[i].cpu().numpy())
                all_m.append(m)

            avg_m = {k: sum(m[k] for m in all_m)/len(all_m) for k in all_m[0]}

            for k in ["policy_loss", "value_loss", "entropy", "mean_bellman_error"]:
                self.metrics_history[k].append(avg_m[k])
            self.metrics_history["mean_episode_return"].append(rollout_m["mean_episode_return"])
            self.metrics_history["mean_episode_rewards"].append(rollout_m["mean_episode_rewards"])

            steps_done += n
            iteration += 1
            obs = last_obs

            if iteration == 1 or iteration % log_every == 0 or steps_done >= total_timesteps:
                print(
                    f"Iter {iteration:4d} | steps={steps_done:>8}/{total_timesteps} | "
                    f"pi={avg_m['policy_loss']:.4f} | v={avg_m['value_loss']:.4f} | "
                    f"ent={avg_m['entropy']:.4f} | bell={avg_m['mean_bellman_error']:.4f} | "
                    f"ret={rollout_m['mean_episode_return']:.4f} | "
                    f"rew={rollout_m['mean_episode_rewards']:.4f}"
                )

    def plot_metrics(self, save_path="ippo_metrics.png", title="IPPO Training Metrics"):
        names = ["policy_loss","value_loss","entropy",
                 "mean_bellman_error","mean_episode_return","mean_episode_rewards"]
        fig, axes = plt.subplots(3, 2, figsize=(14, 12))
        for ax, name in zip(axes.flatten(), names):
            vals = self.metrics_history.get(name, [])
            ax.plot(range(1, len(vals)+1), vals, linewidth=1.8)
            ax.set_title(name); ax.set_xlabel("Iteration"); ax.grid(True, alpha=0.3)
        fig.suptitle(title); plt.tight_layout()
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
        plt.show()
        print(f"Saved: {save_path}")

### Training

In [ ]:
TOTAL_TIMESTEPS = 200_000
ROLLOUT_LENGTH = 2048
SEED = 42
LOG_EVERY = 10

OUTPUT_DIR = "outputs/ippo_guarded_territory"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = get_device()
torch.manual_seed(SEED)
np.random.seed(SEED)

adapter = GuardedTerritoryAdapter(
    num_envs=1, device=device,
    n_scouts=3, n_interceptors=3, n_intruders=3, n_zones=2, max_steps=200,
)

print(f"Device: {device}")
print(f"Defenders: {adapter.n_defenders}, obs_dim: {adapter.obs_dim}, action_dim: {adapter.action_dim}")

trainer = IPPOTrainer(adapter=adapter, hidden_dim=64)
trainer.train(total_timesteps=TOTAL_TIMESTEPS, rollout_length=ROLLOUT_LENGTH, log_every=LOG_EVERY)

plot_path = os.path.join(OUTPUT_DIR, "ippo_guarded_territory_metrics.png")
trainer.plot_metrics(save_path=plot_path, title="IPPO Training (Guarded Territory)")

### Evaluation

In [ ]:
def evaluate_ippo(trainer, episodes=10, max_steps=200, device="cpu"):
    eval_adapter = GuardedTerritoryAdapter(
        num_envs=1, device=device, n_scouts=3, n_interceptors=3,
        n_intruders=3, n_zones=2, max_steps=max_steps,
    )
    episode_returns = []
    step_rewards = []

    for ep in range(episodes):
        obs_b, _ = eval_adapter.reset()
        obs = obs_b.squeeze(0)  # (N, obs_dim)
        ep_return = 0.0

        for _ in range(max_steps):
            actions_np = []
            for i in range(trainer.num_agents):
                act, _, _ = trainer.agents[i].select_action(obs[i])
                actions_np.append(act)

            actions_t = torch.as_tensor(
                np.stack(actions_np), dtype=torch.float32, device=device,
            ).unsqueeze(0)

            obs_b, rew_b, dones_b, _, _ = eval_adapter.step(actions_t)
            obs = obs_b.squeeze(0)
            rewards = rew_b.squeeze(0)

            mean_rew = rewards.mean().item()
            step_rewards.append(mean_rew)
            ep_return += mean_rew

            if dones_b.squeeze(0).item():
                break

        episode_returns.append(ep_return)

    return {
        "episodes": episodes,
        "mean_episode_return": float(np.mean(episode_returns)),
        "std_episode_return": float(np.std(episode_returns)),
        "mean_step_reward": float(np.mean(step_rewards)),
    }


eval_metrics = evaluate_ippo(trainer, episodes=10, device=device)
print("\nEvaluation Results (Guarded Territory, IPPO):")
for k, v in eval_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")